# OpenPlaque — Image-Driven Coronary Tracking + Straightened Plaque Roadmap

This notebook uses an **image-driven 2-D coronary path tracker** on each Siemens curved coronary reformat (CPR) frame.

The path is derived from CT intensity and multiscale vesselness, not from the nnU-Net vessel label. Canonical plaque is displayed only in a narrow tube around the tracked path. The selected CPR path is then straightened into a vessel-centered roadmap.

The workflow is deliberately split into visible steps. Expensive plaque segmentations are reused from validated cache whenever possible; nnU-Net runs only on a true cache miss. Canonical TPV is unchanged. CPR plaque views and source-volume PCAT are not spatially co-registered. Research use only.


## Step 1 — Mount Google Drive


In [ ]:
# FIRST EXECUTABLE CELL — mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Install OpenPlaque and dependencies


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch image-driven-coronary-tracking-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -r /content/OpenPlaque/requirements-colab.txt
import sys
sys.path.insert(0, '/content/OpenPlaque/src')
print('Repository and requirements ready.')


## Step 3 — Prepare inputs and reuse cached plaque masks

This step loads DICOM/metrics, validates cached canonical plaque masks, and only invokes nnU-Net for a vessel whose cache is missing or invalid. Re-running later cells does **not** redo segmentation.


In [ ]:
from openplaque.tracking_workflow import TrackingWorkflow

wf = TrackingWorkflow('/content/drive/MyDrive/OpenPlaque')
series_map = wf.prepare_inputs()
cache_qc = wf.load_cached_plaque()
print('Series:', series_map)
display(cache_qc)


## Step 4 — Track the coronary path in every CPR rotation

Tracking uses CT intensity + multiscale vesselness. The nnU-Net vessel label is not used to define the path.


In [ ]:
tracking_qc = wf.track_coronaries()
display(tracking_qc)


## Step 5 — Review the top tracking candidates

The top three image-driven candidates for LAD, RCA, and LCX are shown so failures are visible rather than hidden by automatic selection.


In [ ]:
candidate_figure = wf.plot_candidates()
print('Saved:', candidate_figure)


## Step 6 — Create straightened vessel-centered plaque roadmaps

Plaque is displayed only inside the 5-mm tube around the selected image-driven path. This is a visualization filter; it does not change canonical TPV.


In [ ]:
roadmap_figure = wf.plot_straightened_roadmaps()
print('Saved:', roadmap_figure)
display(wf.along_df.head(20))


## Step 7 — Reuse validated PCAT figures and build the summary dashboard

The PCAT measurement is not recomputed here. The notebook reuses the previously validated RCA PCAT cross-sections/ribbon and combines them with the tracking QC and canonical quantitative endpoints.


In [ ]:
pcat_sources = wf.reuse_pcat_outputs()
print('Reused PCAT sources:', pcat_sources)
dashboard = wf.plot_dashboard()
print('Saved:', dashboard)


## Step 8 — Package the report-back ZIP


In [ ]:
zip_path = wf.package_report()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_IMAGE_DRIVEN_TRACKING_REPORT_BACK.zip')
